### MFR Demo

In [1]:
import torch

import pandas as pd
from datasets import load_dataset
from utils import counting, mfr, evaluate

import random
import numpy as np

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

g = torch.Generator()
g.manual_seed(SEED)

# Load data
dev_pub_data = load_dataset("weerayut/multilexnorm2026-dev-pub")

# Select data
train, val = dev_pub_data["train"], dev_pub_data["validation"]

c:\Users\DH\Desktop\플래너\5. 학기 자료\26-1\2.인지개\과제\MultiLexNorm2026-main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Submission demo

In [2]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from utils import counting, mfr, evaluate

from train_module import (
    build_label_vocab,
    TokenNormDataset,
    load_trained_model_if_complete,
    train_model,
    predict_with_mfr_fallback,
    evaluate_predictions_by_language,
    evaluate_predictions,
)

In [3]:
print("-----train 파트-----")
print(type(train))
print(len(train))
print(train[0])
print(train.column_names)
print("-----val 파트-----")
print(type(val))
print(len(val))
print(val[0])
print(val.column_names)

-----train 파트-----
<class 'datasets.arrow_dataset.Dataset'>
39178
{'raw': ['Dette', 'er', 'ikke', 'tilfaeldigt', '.'], 'norm': ['Dette', 'er', 'ikke', 'tilfældigt', '.'], 'lang': 'da'}
['raw', 'norm', 'lang']
-----val 파트-----
<class 'datasets.arrow_dataset.Dataset'>
8408
{'raw': ['fänd', 'ich', 'toll', '!', '%PosSmiley'], 'norm': ['Fände', 'ich', 'toll', '!', '%PosSmiley'], 'lang': 'de'}
['raw', 'norm', 'lang']


MFR baseline 재현

In [4]:
train_tok = train
val_tok = val

counts = counting(train_tok)

raw_val = [x["raw"] for x in val_tok]
gold_val = [x["norm"] for x in val_tok]

mfr_pred_val = [mfr(x, counts) for x in raw_val]

lai, acc, err = evaluate(
    raw_val,
    gold_val,
    mfr_pred_val,
    ignCaps=False,
    verbose=False,
    info=True,
)

print("MFR result")
print("LAI:", lai)
print("Accuracy:", acc)
print("ERR:", err)

Baseline acc.(LAI): 88.48
Accuracy:           92.97
ERR:                39.02
MFR result
LAI: 0.8847954569019836
Accuracy: 0.9297478803927355
ERR: 0.3901966214345056


LAI baseline

In [5]:
raw_pred_val = raw_val

lai_raw, acc_raw, err_raw = evaluate(
    raw_val,
    gold_val,
    raw_pred_val,
    ignCaps=False,
    verbose=False,
    info=True,
)

print("Raw baseline result")
print("LAI:", lai_raw)
print("Accuracy:", acc_raw)
print("ERR:", err_raw)

Baseline acc.(LAI): 88.48
Accuracy:           88.48
ERR:                0.00
Raw baseline result
LAI: 0.8847954569019836
Accuracy: 0.8847954569019836
ERR: 0.0


Model / tokenizer 설정

In [6]:
import warnings
warnings.filterwarnings("ignore",category=FutureWarning,module="huggingface_hub.file_download")


MODEL_NAME = "distilbert-base-multilingual-cased"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5

# BATCH_SIZE = 8
# MAX_LEN = 96

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label vocab 생성
-전체 언어를 하나로 학습하므로 train 전체의 norm token을 label vocabulary로 만듦.

In [7]:
label2id, id2label = build_label_vocab(
    train_tok,
    min_freq=1,
    add_unk=True,
)

num_labels = len(label2id)

print("num_labels:", num_labels)

for i, item in enumerate(list(label2id.items())[:5]):
    print(i, item)

num_labels: 115445
0 ('<UNK>', 0)
1 ('_', 1)
2 ('.', 2)
3 (',', 3)
4 ('i', 4)


In [8]:
train_dataset = TokenNormDataset(
    data=train_tok,
    tokenizer=tokenizer,
    label2id=label2id,
    max_len=MAX_LEN,
)
val_dataset = TokenNormDataset(
    data=val_tok,
    tokenizer=tokenizer,
    label2id=label2id,
    max_len=MAX_LEN,
)

DataLoader 생성

In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=g,
    pin_memory=True,)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,)

학습

In [10]:
CKPT_PATH = "token_norm_distilmbert_checkpoint.pt"

model = load_trained_model_if_complete(
    model_name=MODEL_NAME,
    num_labels=num_labels,
    ckpt_path=CKPT_PATH,
    epochs=EPOCHS,
)

if model is None:
    model = train_model(
        train_loader=train_loader,
        val_loader=val_loader,
        model_name=MODEL_NAME,
        num_labels=num_labels,
        epochs=EPOCHS,
        lr=LR,
        ckpt_path=CKPT_PATH,
    )

    
# 모델 단독 예측 평가
# result_model_only = predict_with_mfr_fallback(
#     model=model,
#     tokenizer=tokenizer,
#     data=val_tok,
#     counts=counts,
#     id2label=id2label,
#     max_len=MAX_LEN,
#     use_mfr_fallback=False,
#     conf_threshold=0.0,
#     mfr_ratio_threshold=1.1,
# )

# pred_model_only = result_model_only["pred"]

# print("Model-only evaluation")
# evaluate(
#     raw_val,
#     gold_val,
#     pred_model_only,
#     ignCaps=False,
#     verbose=False,
#     info=True,
# )

🔥 checkpoint 로드: epoch 3부터 시작
✅ checkpoint 학습 완료 상태 감지: start_epoch=3, target_epochs=3
학습 스킵하고 checkpoint 가중치만 사용합니다.


MFR fallback 예측 평가
기본 규칙:
MFR ratio가 높으면 MFR 사용
아니고 model confidence가 threshold 이상이면 model 사용
그 외에는 MFR 사용

In [11]:
# result_fallback = predict_with_mfr_fallback(
#     model=model,
#     tokenizer=tokenizer,
#     data=val_tok,
#     counts=counts,
#     id2label=id2label,
#     max_len=MAX_LEN,
#     use_mfr_fallback=True,
#     conf_threshold=0.80,
#     mfr_ratio_threshold=0.95,
# )

# pred_fallback = result_fallback["pred"]

# print("MFR + classifier fallback evaluation")
# lai_fb, acc_fb, err_fb = evaluate(
#     raw_val,
#     gold_val,
#     pred_fallback,
#     ignCaps=False,
#     verbose=False,
#     info=True,
# )

# print("Fallback result")
# print("LAI:", lai_fb)
# print("Accuracy:", acc_fb)
# print("ERR:", err_fb)

# print("MFR + classifier fallback evaluation by language")

# fallback_eval_by_lang = evaluate_predictions_by_language(
#     data=val_tok,
#     pred=pred_fallback,
#     ignCaps=False,
#     verbose=False,
#     info=True,
# )

threshold 튜닝

In [12]:
# threshold_results = []

# for conf_th in [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]:
# #for conf_th in [0.50, 0.95]:
#     for ratio_th in [0.80, 0.85, 0.90, 0.95, 0.98, 1.01]:
#     #for ratio_th in [0.80, 1.01]:
#         result = predict_with_mfr_fallback(
#             model=model,
#             tokenizer=tokenizer,
#             data=val_tok,
#             counts=counts,
#             id2label=id2label,
#             max_len=MAX_LEN,
#             use_mfr_fallback=True,
#             conf_threshold=conf_th,
#             mfr_ratio_threshold=ratio_th,
#         )

#         pred = result["pred"]

#         lai, acc, err = evaluate(
#             raw_val,
#             gold_val,
#             pred,
#             ignCaps=False,
#             verbose=False,
#             info=False,
#         )

#         threshold_results.append({
#             "conf_threshold": conf_th,
#             "mfr_ratio_threshold": ratio_th,
#             "lai": lai,
#             "accuracy": acc,
#             "err": err,
#         })

# threshold_df = pd.DataFrame(threshold_results)
# threshold_df = threshold_df.sort_values("err", ascending=False)

# threshold_df.head(20)

best threshold로 재평가

In [13]:
# best = threshold_df.iloc[0]

# best_conf_th = float(best["conf_threshold"])
# best_ratio_th = float(best["mfr_ratio_threshold"])

# print("best_conf_threshold:", best_conf_th)
# print("best_mfr_ratio_threshold:", best_ratio_th)
# print(best)

In [14]:
# best_result = predict_with_mfr_fallback(
#     model=model,
#     tokenizer=tokenizer,
#     data=val_tok,
#     counts=counts,
#     id2label=id2label,
#     max_len=MAX_LEN,
#     use_mfr_fallback=True,
#     conf_threshold=best_conf_th,
#     mfr_ratio_threshold=best_ratio_th,
# )

# best_pred = best_result["pred"]

# evaluate(
#     raw_val,
#     gold_val,
#     best_pred,
#     ignCaps=False,
#     verbose=False,
#     info=True,
# )

어떤 token이 MFR에서 모델로 바뀌었는지 확인

In [15]:
# changes = []

# for item, mfr_sent, pred_sent in zip(val_tok, mfr_pred_val, best_pred):
#     raw_sent = item["raw"]
#     gold_sent = item["norm"]

#     for raw_tok, gold_tok, mfr_tok, pred_tok in zip(
#         raw_sent,
#         gold_sent,
#         mfr_sent,
#         pred_sent,
#     ):
#         if mfr_tok != pred_tok:
#             changes.append({
#                 "raw": raw_tok,
#                 "gold": gold_tok,
#                 "mfr": mfr_tok,
#                 "pred": pred_tok,
#                 "mfr_correct": mfr_tok == gold_tok,
#                 "pred_correct": pred_tok == gold_tok,
#             })

# changes_df = pd.DataFrame(changes)

# print("num changed from MFR:", len(changes_df))
# changes_df.head(50)

MFR보다 좋아진/나빠진 케이스 확인

In [16]:
# if len(changes_df) > 0:
#     print("Model fixed MFR errors:")
#     display(changes_df[(changes_df["mfr_correct"] == False) & (changes_df["pred_correct"] == True)].head(50))

#     print("Model broke MFR correct predictions:")
#     display(changes_df[(changes_df["mfr_correct"] == True) & (changes_df["pred_correct"] == False)].head(50))
# else:
#     print("No differences from MFR.")

prediction 저장

In [17]:
# output_df = pd.DataFrame({
#     "raw": raw_val,
#     "norm": gold_val,
#     "mfr_pred": mfr_pred_val,
#     "pred": best_pred,
# })

# output_df.to_pickle("val_predictions.pkl")
# output_df.head()

# codabench 제출용

In [ ]:
data, save_path = load_dataset("weerayut/multilexnorm2026-dev-pub"), "outputs/submission_dev"

def prediction(train, test):
    # 1. train + validation 전체 기준 MFR counts 만들기
    train_df = train.to_pandas()
    counts_submit = counting(train_df.to_dict(orient="records"))

    # 2. test는 그대로 사용
    test_tok = test

    # 3. 모델 + MFR fallback 예측
    result = predict_with_mfr_fallback(
        model=model,
        tokenizer=tokenizer,
        data=test_tok,
        counts=counts_submit,
        id2label=id2label,
        max_len=MAX_LEN,
        use_mfr_fallback=True,
        conf_threshold=0.5,
        mfr_ratio_threshold=0.8,
    )

    # 4. 제출 형식
    test_df = test.to_pandas()
    test_df["pred"] = result["pred"]

    return test_df

In [19]:
from datasets import concatenate_datasets

train_submit = concatenate_datasets([data["train"], data["validation"]])

out = prediction(train_submit, data["test"])

out.to_json(
    f"{save_path}/predictions.json",
    orient="records",
    force_ascii=False
)

out.head()

NameError: name 'best_conf_th' is not defined